# 1. Price Chart + Buy Signals (1 trade per day enforced)


In [4]:
!pip install ta

  Using cached ta-0.11.0-py3-none-any.whl

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [8]:
!pip install loguru


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
from app.indicators import Strategy
from app.risk_management import RiskManagement

def backtest_engine(data_15m_raw, data_1h_raw, data_1d_raw, initial_investment=1000):
    trades = []
      # Ensure datetime format for open_time
    data_15m_raw['open_time'] = pd.to_datetime(data_15m_raw['open_time'])
    data_1h_raw['open_time'] = pd.to_datetime(data_1h_raw['open_time'])
    data_1d_raw['open_time'] = pd.to_datetime(data_1d_raw['open_time'])

    # Step 1: Generate 1D Bias signals
    strat_1d = Strategy(data_1d_raw, timeframe_type='1d')
    data_1d = strat_1d.generate_signals()
    data_1d = data_1d[data_1d['Bias'] == 1]

    for _, row_1d in data_1d.iterrows():
        time_1d = row_1d['timestamp']

        # Step 2: Apply strategy only on 1H data AFTER 1D bias signal
        data_1h_forward = data_1h_raw[data_1h_raw['open_time'] > time_1d]
        strat_1h = Strategy(data_1h_forward, timeframe_type='1h')
        data_1h = strat_1h.generate_signals()
        data_1h = data_1h[data_1h['Confirm'] == 1]

        if data_1h.empty:
            continue

        for _, row_1h in data_1h.iterrows():
            time_1h = row_1h['timestamp']

            # Step 3: Apply strategy only on 15m data AFTER 1H confirm signal
            data_15m_forward = data_15m_raw[data_15m_raw['open_time'] > time_1h]
            strat_15m = Strategy(data_15m_forward, timeframe_type='15m')
            data_15m = strat_15m.generate_signals()
            data_15m = data_15m[data_15m['Entry'] == 1]

            if data_15m.empty:
                continue

            for _, row_15 in data_15m.iterrows():
                entry_time = row_15['timestamp']
                entry_price = row_15['close']
                atr = row_15.get('atr', 0)

                # Risk manager
                risk_mgr = RiskManagement(
                    priceorder=entry_price,
                    currentprice=entry_price,
                    target_profit=2,  # 2% TP
                    stoploss=1,       # 1% SL
                    dollar_investment=initial_investment,
                    atr=atr
                )

                # Monitor for exit from this point forward
                exit_candidates = data_15m_raw[data_15m_raw['open_time'] > entry_time]
                for _, exit_row in exit_candidates.iterrows():
                    current_price = exit_row['close']
                    exit_time = pd.to_datetime(exit_row['open_time'])

                    risk_mgr.currentprice = current_price
                    if risk_mgr.should_exit():
                        profit_loss = risk_mgr.profit_or_loss or 0
                        duration_mins = (exit_time - entry_time).total_seconds() / 60

                        trades.append({
                            'entry_time': entry_time,
                            'exit_time': exit_time,
                            'entry_price': entry_price,
                            'exit_price': current_price,
                            'profit_loss': profit_loss,
                            'duration_mins': duration_mins
                        })
                        print(f"Trade: Entry @ {entry_time}, Exit @ {exit_time}, P/L: {profit_loss:.2f}")
                        break  # Exit 15m loop (this trade is done)
                break  # Exit 1H loop after trade
            break  # Exit 1D loop after trade

    return df_trades
